# Mission1

In [2]:
import os
import json
from PIL import Image

def convert_to_yolo(json_path, img_path, save_path, class_id=0):
    with open(json_path, "r") as f:
        data = json.load(f)
    
    # bbox 정보
    x = data["shape_attributes"]["x"]
    y = data["shape_attributes"]["y"]
    w = data["shape_attributes"]["width"]
    h = data["shape_attributes"]["height"]

    # 이미지 크기
    img = Image.open(img_path)
    img_w, img_h = img.size

    # YOLO 형식 변환
    x_center = (x + w/2) / img_w
    y_center = (y + h/2) / img_h
    w_norm = w / img_w
    h_norm = h / img_h

    # 저장
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    with open(save_path, "w") as f:
        f.write(f"{class_id} {x_center:.6f} {y_center:.6f} {w_norm:.6f} {h_norm:.6f}\n")

# 예시
convert_to_yolo('53.대기오염 배출원 공간 분포 데이터/3.개방데이터/1.데이터/Training/01.원천데이터/TS_KS')

TypeError: convert_to_yolo() missing 2 required positional arguments: 'img_path' and 'save_path'

In [3]:
!pip install ultralytics

from ultralytics import YOLO

# 모델 불러오기 (pretrained)
model = YOLO("yolov8n.pt")  # n=nano, s=small, m=medium, l=large

# 학습
model.train(
    data="data.yaml",   # dataset config (train/val 경로, class 이름)
    epochs=50,
    imgsz=640,
    batch=16,
    name="chimney_yolo"
)

# 검증
metrics = model.val()
print(metrics)

zsh:1: command not found: pip


ModuleNotFoundError: No module named 'ultralytics'

# Mission2

In [1]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms
from PIL import Image
from sklearn.metrics import mean_squared_error

# ======================
# 1. 데이터셋 정의
# ======================
class ChimneyDataset(Dataset):
    def __init__(self, img_dir, label_dir, transform=None):
        self.img_dir = img_dir
        self.label_dir = label_dir
        self.transform = transform
        self.img_files = sorted(os.listdir(img_dir))
        self.label_files = sorted(os.listdir(label_dir))

        # 파일명 매칭
        self.file_names = [f for f in self.img_files if f.replace(".jpg",".json") in self.label_files]

    def __len__(self):
        return len(self.file_names)

    def __getitem__(self, idx):
        fname = self.file_names[idx]
        img_path = os.path.join(self.img_dir, fname)
        label_path = os.path.join(self.label_dir, fname.replace(".jpg", ".json"))

        # 이미지 로드
        img = Image.open(img_path).convert("RGB")
        if self.transform:
            img = self.transform(img)

        # JSON에서 높이 값 로드
        with open(label_path, "r") as f:
            label_json = json.load(f)
            height = float(label_json["region_attributes"]["chi_height_m"])

        return img, torch.tensor(height, dtype=torch.float32)

# ======================
# 2. 데이터 변환 (전처리)
# ======================
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225])
])

train_dataset = ChimneyDataset(
    "53.대기오염 배출원 공간 분포 데이터/Training/TS_KS",
    "53.대기오염 배출원 공간 분포 데이터/Training/TL_KS_LINE",
    transform=transform
)

val_dataset = ChimneyDataset(
    "53.대기오염 배출원 공간 분포 데이터/Validation/VS_KS",
    "53.대기오염 배출원 공간 분포 데이터/Validation/VL_KS_LINE",
    transform=transform
)

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)

# ======================
# 3. 모델 정의 (ResNet18 + Regression)
# ======================
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")
model = models.resnet18(pretrained=True)
model.fc = nn.Linear(model.fc.in_features, 1)  # 회귀 출력
model = model.to(device)

criterion = nn.MSELoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

# ======================
# 4. 학습 루프
# ======================
EPOCHS = 10
for epoch in range(EPOCHS):
    model.train()
    train_loss = 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)

        optimizer.zero_grad()
        outputs = model(imgs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        train_loss += loss.item()

    # Validation
    model.eval()
    preds, trues = [], []
    with torch.no_grad():
        for imgs, labels in val_loader:
            imgs, labels = imgs.to(device), labels.to(device).unsqueeze(1)
            outputs = model(imgs)
            preds.extend(outputs.squeeze().cpu().numpy())
            trues.extend(labels.squeeze().cpu().numpy())

    rmse = mean_squared_error(trues, preds, squared=False)

    print(f"Epoch [{epoch+1}/{EPOCHS}] "
          f"Train Loss: {train_loss/len(train_loader):.4f} "
          f"Val RMSE: {rmse:.4f}")

# ======================
# 5. 모델 저장
# ======================
torch.save(model.state_dict(), "chimney_height_resnet18.pth")

FileNotFoundError: [Errno 2] No such file or directory: '53.대기오염 배출원 공간 분포 데이터/Training/TS_KS'